In [ ]:
import torch

import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau, CosineAnnealingLR
import numpy as np
from tqdm.auto import tqdm
from src.compute_text_representations import compute_text_representations
import pandas as pd
from src.pairwise_dataset import PairwiseDataset
import plotly.express as px
from sklearn.manifold import MDS, TSNE
from sklearn.decomposition import PCA
from umap import UMAP
from sklearn.linear_model import LinearRegression

torch.set_float32_matmul_precision("medium")

In [ ]:
df = pd.read_csv("datasets/short_sentence.csv")
sentences = df["sentence"].tolist()
df = df.drop(columns="sentence")
embeddings = compute_text_representations(
    sentences, model_name="bert-base-uncased", token_aggregation="mean"
)
Y = embeddings[5]

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler


def pca_after_removing_confounder(X, confounder, n_components=2):
    """
    Performs PCA after removing the linear effect of a confounder.

    Args:
        X (np.ndarray): Data matrix (n_samples, n_features), e.g., sentence embeddings.
        confounder (np.ndarray): Confounding variable (n_samples,), e.g., sentence lengths.
        n_components (int): Number of principal components to return.

    Returns:
        np.ndarray: Data projected onto principal components after removing confounder effect.
        PCA: Fitted PCA object.
    """
    n_samples, _ = X.shape
    if confounder.shape[0] != n_samples:
        raise ValueError(
            "X and confounder must have the same number of samples."
        )

    confounder = confounder.reshape(-1, 1)
    scaler = StandardScaler(with_std=False)  # Only mean-center
    X = scaler.fit_transform(X)
    confounder = scaler.fit_transform(confounder)
    regression = LinearRegression()
    regression.fit(confounder, X)
    X_proj_confounder = regression.predict(confounder)
    X -= X_proj_confounder
    pca = PCA(n_components=n_components)
    X_pca_orthogonal = pca.fit_transform(X)

    return X_pca_orthogonal

In [ ]:
lengths = np.array([len(sentence.split()) for sentence in sentences])

In [ ]:
proj = pca_after_removing_confounder(Y, lengths, n_components=2)

In [ ]:
device = "cuda"
Y = Y.to(device)

In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler, MinMaxScaler

In [ ]:
mlp = MLPRegressor(hidden_layer_sizes=tuple(), max_iter=1000)
lengths = df[["sentence_length"]]
lengths = StandardScaler().fit_transform(MinMaxScaler().fit_transform(lengths))
mlp.fit(lengths, Y)
Y -= mlp.predict(lengths)

In [ ]:
Y = StandardScaler().fit_transform(Y)

In [ ]:
dataset = PairwiseDataset(Y=Y, gamma=1.05)

In [ ]:
# %%time
proj = MDS(n_components=2, verbose=2, n_jobs=-2, eps=0.01).fit_transform(
    Y.cpu().numpy()
)

In [ ]:
proj = PCA(3).fit_transform(Y)

In [ ]:
proj = TSNE(2).fit_transform(Y.cpu().numpy())

In [ ]:
length = np.array([len(s.split()) for s in sentences])

In [ ]:
lr = LinearRegression()
lr.fit(Y, length)
# Y -= lr.predict(length)
# proj = PCA(2).fit_transform(Y)

In [ ]:
lr.coef_.shape

In [ ]:
proj = UMAP(2).fit_transform(Y.cpu().numpy())

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

lda = LinearDiscriminantAnalysis(n_components=2)
proj = lda.fit(Y.cpu(), df.sentence_length).transform(Y.cpu())

In [ ]:
# %%time
n_points = Y.shape[0]
n_components = 2
embeddings = torch.randn(
    n_points, n_components, device=device, requires_grad=True
)
optimizer = optim.AdamW([embeddings], lr=1)
scheduler = ReduceLROnPlateau(optimizer)
best_loss = torch.inf
patience = 0
losses = []
for i in (pbar := tqdm(range(10000), desc="Fitting MDS")):
    idx1, idx2, target_dist = dataset.sample(
        int(4096 * 64 * (1**i)), get_idx=True
    )
    optimizer.zero_grad(set_to_none=True)
    x = embeddings[idx1]
    y = embeddings[idx2]
    embedded_dist = (x - y).norm(dim=1, p=2)
    loss = ((embedded_dist - target_dist).pow(2)).mean()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    scheduler.step(loss.item())
    norm = embeddings.grad.detach().norm(p=2)
    pbar.set_postfix(
        loss=f"{loss:.4f}",
        lr=f"{optimizer.param_groups[0]['lr']:.1e}",
        norm=f"{norm:.4f}",
    )
    if loss < best_loss:
        best_loss = loss.item()
        patience = 0
    else:
        patience += 1

    if patience >= 20:
        break

In [ ]:
px.line(losses)

In [ ]:
px.scatter(
    embeddings.detach().cpu().numpy(),
    x=0,
    y=1,
    color=df.sentence_length,
    hover_name=sentences,
)

In [ ]:
px.scatter(proj, x=0, y=1, color=df.index, hover_name=sentences)

In [ ]:
df.columns

In [ ]:
proj = pca_after_removing_confounder(Y, lengths, 3)

In [ ]:
px.scatter(
    proj,
    x=0,
    y=1,
    color=df.sentence_GROUP + lengths.astype(str),
    hover_name=sentences,
)